In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta

# 경로 설정
target_dir = "/mnt/AEGIS/AEGIS/Deidentification/AEGIS-Basic-Order/de"

# 환자별 노이즈 정보
noise_df = pd.read_csv("/mnt/AEGIS/AEGIS/Deidentification/AEGIS-PT/pt_noise_date_time.csv", sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
noise_df['new_pt_id'] = noise_df['new_pt_id'].astype(str)

pt_id_map = noise_df.set_index('pt_id')['new_pt_id'].to_dict()
date_noise_map = noise_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_noise_map = noise_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(target_dir, '**/*.csv'), recursive = True)
print(f"총 처리 대상 파일 수 : {len(csv_files)}")



In [ ]:
# 컬럼 삭제
print("\n 컬럼 제거")

columns_to_drop = [
    'user_cd', 'user', 'tx_center_cd', 'tx_center_nm', 'ward_cd', 'ward_nm', 'certification_cd', 'certification', 'certification_dt'
]

for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        total_cols_to_remove = len(columns_to_drop)
        cols_in_df = [col for col in columns_to_drop if col in df.columns]
        removed_cols_count = len(cols_in_df)
        
        if removed_cols_count > 0:
            df.drop(columns = cols_in_df, inplace = True)
            
        df.to_csv(file, index=False, encoding = 'utf-8')
        
        print(f"{file} - 제거 대상: {total_cols_to_remove}개, 실제 제거: {removed_cols_count}개")
    
    except Exception as e:
        print(f"[❗ hos_dt 오류] {file} → {e}")

In [ ]:
# opd_nm, depart, ad_ward에서 신부인과, 정신건강의학과 제거

import os
import glob
import pandas as pd


# 키워드 정의
keywords = "산부인과|정신건강의학과|정신의학과"

# 처리 시작
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        original_len = len(df)

        # 각 컬럼에 대해 조건에 맞는 행 제거
        condition = pd.Series([False] * len(df))
        for col in ['depart_nm']:
            if col in df.columns:
                condition = condition | df[col].fillna('').str.contains(keywords, regex=True)

        # 해당 행 제거
        df = df[~condition]
        removed = original_len - len(df)

        # 덮어쓰기 저장
        df.to_csv(file, index=False, encoding='utf-8')
        print(f"✅ {os.path.basename(file)}: {removed}건 제거 완료")

    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")


In [ ]:
# ① pt_id 치환
print("\n① pt_id 치환 시작")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')
        
        if 'pt_id' in df.columns:
            df['pt_id'] = df['pt_id'].map(pt_id_map).fillna(df['pt_id'])
            
        print(df.head(3))
        
        df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"✅ pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ pt_id 오류] {file} → {e}")

In [ ]:
# ② hos_dt ±7일 ±30분
print("\n② hos_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'hos_dt' in df.columns:
            df['hos_dt'] = pd.to_datetime(df['hos_dt'], errors='coerce')
            df['hos_dt'] = df['hos_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ hos_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ hos_dt 오류] {file} → {e}")



In [ ]:
# order_date 처리

for file in csv_files:

    try:

        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        

        if 'order_date' in df.columns:

            df['order_date'] = pd.to_datetime(df['order_date'], errors = 'coerce')

            

            df['order_date'] = df['order_date'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D")
            

            print(df.head(3))

            

            df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"order_date 완료: {file}")

            

    

    except Exception as e:

        print(f"[order_date 오류] {file} -> {e}")

In [ ]:
# order_finish_dt ±7일 ±30분
print("\n order_finish_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'order_finish_dt' in df.columns:
            df['order_finish_dt'] = pd.to_datetime(df['order_finish_dt'], errors='coerce')
            df['order_finish_dt'] = df['order_finish_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ order_finish_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ order_finish_dt 오류] {file} → {e}")



In [ ]:
# order_confirm_dt ±7일 ±30분
print("\norder_confirm_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'order_confirm_dt' in df.columns:
            df['order_confirm_dt'] = pd.to_datetime(df['order_confirm_dt'], errors='coerce')
            df['order_confirm_dt'] = df['order_confirm_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ order_confirm_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ order_confirm_dt 오류] {file} → {e}")



In [ ]:
# tx_dt ±7일 ±30분
print("\ntx_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'tx_dt' in df.columns:
            df['tx_dt'] = pd.to_datetime(df['tx_dt'], errors='coerce')
            df['tx_dt'] = df['tx_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ tx_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ tx_dt 오류] {file} → {e}")



In [ ]:
# first_dt ±7일 ±30분
print("\nfirst_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'first_dt' in df.columns:
            df['first_dt'] = pd.to_datetime(df['first_dt'], errors='coerce')
            df['first_dt'] = df['first_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ first_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ first_dt 오류] {file} → {e}")

